In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [ ]:
import pandas as pd
import numpy as np
import transformers
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from tqdm.notebook import tqdm
import random

# Custom F1 score function
def compute_f1(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'f1': f1_score(labels, predictions, average='binary')
    }

# Load and prepare data
df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# For initial testing, use 10% of data
subset_df, _ = train_test_split(df, train_size=0.1, stratify=df['label'], random_state=42)
train, val = train_test_split(subset_df, test_size=0.2, stratify=subset_df['label'], random_state=42)

# Convert to datasets
train_ds = Dataset.from_pandas(train)
val_ds = Dataset.from_pandas(val)

# Models to test
models = [
    'bert-base-uncased',
    'roberta-base',
    'microsoft/deberta-base',
    'google/electra-base-discriminator',
    'distilbert-base-uncased'
]

results = {}

def tokenize(batch):
    return tokenizer(batch['review'], padding=True, truncation=True, max_length=512)

# Training arguments
training_args = transformers.TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',  # Updated from evaluation_strategy
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1'
)

# Train and evaluate each model
for model_name in models:
    print(f"\nTraining {model_name}...")
    
    # Initialize tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    
    # Tokenize datasets
    train_ds_tokenized = train_ds.map(tokenize, batched=True)
    val_ds_tokenized = val_ds.map(tokenize, batched=True)
    
    train_ds_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    val_ds_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    
    # Initialize trainer
    trainer = transformers.Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds_tokenized,
        eval_dataset=val_ds_tokenized,
        tokenizer=tokenizer,
        compute_metrics=compute_f1
    )
    
    # Train model
    trainer.train()
    
    # Evaluate and store results
    eval_results = trainer.evaluate()
    results[model_name] = eval_results['eval_f1']
    
    # Save model
    trainer.save_model(f'./results/{model_name.split("/")[-1]}')

# Find best model
best_model = max(results, key=results.get)
print(f"\nBest model: {best_model} with F1 score: {results[best_model]}")

# Finetune best model on full dataset
print(f"\nFinetuning {best_model} on full dataset...")
train_full, val_full = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_ds_full = Dataset.from_pandas(train_full)
val_ds_full = Dataset.from_pandas(val_full)

tokenizer = AutoTokenizer.from_pretrained(best_model)
model = AutoModelForSequenceClassification.from_pretrained(best_model, num_labels=2)

train_ds_full = train_ds_full.map(tokenize, batched=True)
val_ds_full = val_ds_full.map(tokenize, batched=True)

train_ds_full.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds_full.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_full,
    eval_dataset=val_ds_full,
    tokenizer=tokenizer,
    compute_metrics=compute_f1
)

trainer.train()
trainer.save_model(f'./results/{best_model.split("/")[-1]}_full')

# Run inference on 10 random samples
test_samples = df.sample(n=10, random_state=42)
clf_pipeline = pipeline(
    'text-classification',
    model=f'./results/{best_model.split("/")[-1]}_full',
    device=0 if torch.cuda.is_available() else -1
)

print("\nInference results:")
for idx, row in test_samples.iterrows():
    result = clf_pipeline(row['review'][:512])[0]  # Truncate to 512 tokens
    predicted = 'positive' if result['label'] == 'LABEL_1' else 'negative'
    actual = 'positive' if row['label'] == 1 else 'negative'
    print(f"\nSample {idx}:")
    print(f"Review (truncated): {row['review'][:100]}...")
    print(f"Predicted: {predicted}, Actual: {actual}, Confidence: {result['score']:.3f}")


Training bert-base-uncased...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_36/169476426.py:78: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = transformers.Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>